# AI-Powered GPI System 快速入门

本 Notebook 演示如何使用系统的各个模块进行游戏玩家反馈分析。

## 1. 环境准备

In [ ]:
import sys
sys.path.append('..')

import pandas as pd
import yaml
import logging

logging.basicConfig(level=logging.INFO)

## 2. 加载配置

In [ ]:
with open('../config/config.yaml', 'r', encoding='utf-8') as f:
    config = yaml.safe_load(f)

print("配置加载成功")

## 3. 数据采集（Steam 评论）

In [ ]:
from src.data_collection.steam_scraper import SteamReviewScraper

# 创建爬虫实例
scraper = SteamReviewScraper(config)

# 采集评论（以 Cyberpunk 2077 为例）
game_id = 1091500
game_name = "Cyberpunk_2077"
max_reviews = 1000  # 示例，采集1000条评论

raw_reviews = scraper.get_reviews(game_id, max_reviews=max_reviews)
df_raw = scraper.parse_reviews(raw_reviews)

print(f"成功采集 {len(df_raw)} 条评论")
df_raw.head()

## 4. 数据清洗

In [ ]:
from src.data_processing.data_cleaner import DataCleaner

# 创建清洗器
cleaner = DataCleaner(config)

# 执行清洗
df_clean = cleaner.clean(df_raw)
df_clean = cleaner.add_features(df_clean)

print(f"清洗后数据量: {len(df_clean)} 条")
print(f"\n玩家类型分布:")
print(df_clean['player_type'].value_counts())

## 5. 情绪分析

In [ ]:
from src.analysis.sentiment.sentiment_analyzer import SentimentAnalyzer

# 创建情绪分析器
sentiment_analyzer = SentimentAnalyzer(config)

# 执行情绪分析
df_sentiment = sentiment_analyzer.analyze_dataframe(df_clean)

# 获取统计信息
stats = sentiment_analyzer.get_sentiment_stats(df_sentiment)

print("\n情绪分布:")
for sentiment, pct in stats['sentiment_percentage'].items():
    print(f"  {sentiment.capitalize()}: {pct:.1%}")

## 6. 主题分析

In [ ]:
from src.analysis.topic.topic_analyzer import TopicAnalyzer

# 创建主题分析器
topic_analyzer = TopicAnalyzer(config)

# 执行主题分析
df_topic, topics = topic_analyzer.analyze_dataframe(df_sentiment)

print("\n主要讨论主题:")
for topic_id, words in list(topics.items())[:5]:
    print(f"主题 {topic_id}: {', '.join(words[:5])}")

## 7. AI 评论总结

In [ ]:
from src.ai_modules.summarizer.review_summarizer import ReviewSummarizer

# 创建总结器
summarizer = ReviewSummarizer(config)

# 按情绪总结评论
summaries = summarizer.summarize_by_sentiment(df_topic)

print("\n=== 负面评论总结 ===")
print(summaries.get('negative', '无数据'))

## 8. 生成运营策略

In [ ]:
from src.ai_modules.strategy.strategy_generator import StrategyGenerator

# 创建策略生成器
strategy_gen = StrategyGenerator(config)

# 准备分析结果
analysis_results = {
    'sentiment_stats': stats,
    'topics': topics,
    'negative_summary': summaries.get('negative', '')
}

# 生成策略
strategies = strategy_gen.generate_strategies(analysis_results, {'name': game_name})

print("\n=== 运营策略建议 ===")
print(strategies)

## 9. 可视化分析

In [ ]:
import plotly.express as px

# 情绪分布饼图
sentiment_counts = df_topic['sentiment'].value_counts()
fig = px.pie(
    values=sentiment_counts.values,
    names=sentiment_counts.index,
    title='玩家情绪分布'
)
fig.show()

# 玩家类型分布
player_counts = df_topic['player_type'].value_counts()
fig = px.bar(
    x=player_counts.index,
    y=player_counts.values,
    title='玩家类型分布',
    labels={'x': '玩家类型', 'y': '数量'}
)
fig.show()

## 10. 保存结果

In [ ]:
# 保存分析结果
df_topic.to_csv(f'../data/results/{game_name}_analysis_result.csv', index=False, encoding='utf-8-sig')

print("\n✅ 分析完成！结果已保存。")